# Assignment 9

## Imports

In [3]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

---
## Task 1 [3pkt]

1. Download the dataset from the following link:  
   https://www.kaggle.com/datasets/humansintheloop/semantic-segmentation-of-aerial-imagery  
2. Take a first look at the data. Are the classes balanced? Visualize a few interesting examples to better understand the dataset.  
3. Build a UNet model in PyTorch. You can use standard layers like `nn.Conv2d`, `nn.MaxPool2d`, and `nn.ConvTranspose2d`. Use Cross Entropy (`nn.CrossEntropyLoss`) as your loss function. Then train the model using the downloaded dataset.  
4. After training, visualize some filters from the convolutional layers. Do they seem to detect useful patterns or features?  
5. Evaluate the results. Does the model perform better on certain classes? Are there classes that are harder to predict?

In [1]:
!kaggle datasets download -d humansintheloop/semantic-segmentation-of-aerial-imagery
!unzip semantic-segmentation-of-aerial-imagery.zip -d dataset/

Dataset URL: https://www.kaggle.com/datasets/humansintheloop/semantic-segmentation-of-aerial-imagery
License(s): CC0-1.0
 91%|██████████████████████████████████▋   | 27.0M/29.6M [00:01<00:00, 24.9MB/s]
100%|██████████████████████████████████████| 29.6M/29.6M [00:01<00:00, 18.1MB/s]
Archive:  semantic-segmentation-of-aerial-imagery.zip
  inflating: dataset/Semantic segmentation dataset/Tile 1/images/image_part_001.jpg  
  inflating: dataset/Semantic segmentation dataset/Tile 1/images/image_part_002.jpg  
  inflating: dataset/Semantic segmentation dataset/Tile 1/images/image_part_003.jpg  
  inflating: dataset/Semantic segmentation dataset/Tile 1/images/image_part_004.jpg  
  inflating: dataset/Semantic segmentation dataset/Tile 1/images/image_part_005.jpg  
  inflating: dataset/Semantic segmentation dataset/Tile 1/images/image_part_006.jpg  
  inflating: dataset/Semantic segmentation dataset/Tile 1/images/image_part_007.jpg  
  inflating: dataset/Semantic segmentation dataset/Tile 1/ima

In [4]:
DATASET_ROOT = Path("dataset")  # adjust if needed

# Class definitions (Dubai Aerial Imagery dataset)
CLASSES = {
    0: "Building",
    1: "Land (unpaved)",
    2: "Road",
    3: "Vegetation",
    4: "Water",
    5: "Unlabeled",
}

# Color palette used in the masks (RGB)
PALETTE = {
    (60, 16, 152): 0,   # Building
    (132, 41, 246): 1,  # Land
    (110, 193, 228): 2, # Road
    (254, 221, 58): 3,  # Vegetation
    (226, 169, 41): 4,  # Water
    (155, 155, 155): 5, # Unlabeled
}

NUM_CLASSES = len(CLASSES)
print("Classes:", CLASSES)

Classes: {0: 'Building', 1: 'Land (unpaved)', 2: 'Road', 3: 'Vegetation', 4: 'Water', 5: 'Unlabeled'}


### 1.2 Class Balance & Visualization

In [ ]:
def rgb_mask_to_class(mask_rgb: np.ndarray) -> np.ndarray:
    """Convert RGB mask image to single-channel class index array."""
    h, w, _ = mask_rgb.shape
    class_mask = np.zeros((h, w), dtype=np.uint8)
    for rgb, cls_idx in PALETTE.items():
        match = np.all(mask_rgb == np.array(rgb), axis=-1)
        class_mask[match] = cls_idx
    return class_mask


def collect_image_mask_pairs(root: Path):
    """Recursively find (image, mask) path pairs."""
    pairs = []
    for img_path in sorted(root.rglob("images/*.jpg")):
        mask_path = img_path.parent.parent / "masks" / (img_path.stem + ".png")
        if mask_path.exists():
            pairs.append((img_path, mask_path))
    return pairs


pairs = collect_image_mask_pairs(DATASET_ROOT)
print(f"Found {len(pairs)} image-mask pairs")

In [ ]:
# Count pixel distribution per class
class_counts = Counter()
for _, mask_path in pairs:
    mask_rgb = np.array(Image.open(mask_path).convert("RGB"))
    class_mask = rgb_mask_to_class(mask_rgb)
    class_counts.update(Counter(class_mask.flatten().tolist()))

labels = [CLASSES[i] for i in range(NUM_CLASSES)]
counts = [class_counts.get(i, 0) for i in range(NUM_CLASSES)]

plt.figure(figsize=(10, 5))
plt.bar(labels, counts)
plt.title("Pixel count per class")
plt.ylabel("Pixel count")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

total = sum(counts)
for cls, cnt in zip(labels, counts):
    print(f"{cls}: {cnt:,} pixels ({100*cnt/total:.1f}%)")

In [ ]:
# Visualize a few examples
sample_pairs = random.sample(pairs, min(4, len(pairs)))

fig, axes = plt.subplots(len(sample_pairs), 2, figsize=(12, 4 * len(sample_pairs)))
for row, (img_path, mask_path) in enumerate(sample_pairs):
    img = np.array(Image.open(img_path).convert("RGB"))
    mask_rgb = np.array(Image.open(mask_path).convert("RGB"))
    axes[row, 0].imshow(img)
    axes[row, 0].set_title(f"Image: {img_path.name}")
    axes[row, 0].axis("off")
    axes[row, 1].imshow(mask_rgb)
    axes[row, 1].set_title("Mask")
    axes[row, 1].axis("off")
plt.tight_layout()
plt.show()

**Are the classes balanced?**

> *Your answer here*

### 1.3 Dataset & DataLoader

In [ ]:
IMG_SIZE = 256


class AerialDataset(Dataset):
    def __init__(self, pairs, img_size=IMG_SIZE, augment=False):
        self.pairs = pairs
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img = Image.open(img_path).convert("RGB").resize((self.img_size, self.img_size))
        mask_rgb = np.array(Image.open(mask_path).convert("RGB").resize(
            (self.img_size, self.img_size), Image.NEAREST))
        mask = rgb_mask_to_class(mask_rgb)

        if self.augment and random.random() > 0.5:
            img = TF.hflip(img)
            mask = np.fliplr(mask).copy()

        img_tensor = transforms.ToTensor()(img)  # [3, H, W]
        mask_tensor = torch.from_numpy(mask).long()  # [H, W]
        return img_tensor, mask_tensor


random.shuffle(pairs)
split = int(0.8 * len(pairs))
train_pairs, val_pairs = pairs[:split], pairs[split:]

train_ds = AerialDataset(train_pairs, augment=True)
val_ds = AerialDataset(val_pairs, augment=False)

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=8, shuffle=False, num_workers=2)

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")

### 1.4 UNet Model

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class UNet(nn.Module):
    def __init__(self, in_channels=3, num_classes=6, features=(64, 128, 256, 512)):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pools = nn.ModuleList()
        self.decoders = nn.ModuleList()
        self.upconvs = nn.ModuleList()

        # Encoder
        ch = in_channels
        for f in features:
            self.encoders.append(DoubleConv(ch, f))
            self.pools.append(nn.MaxPool2d(2))
            ch = f

        # Bottleneck
        self.bottleneck = DoubleConv(features[-1], features[-1] * 2)

        # Decoder
        for f in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(f * 2, f, 2, stride=2))
            self.decoders.append(DoubleConv(f * 2, f))

        self.final = nn.Conv2d(features[0], num_classes, 1)

    def forward(self, x):
        skip_connections = []
        for enc, pool in zip(self.encoders, self.pools):
            x = enc(x)
            skip_connections.append(x)
            x = pool(x)

        x = self.bottleneck(x)

        for up, dec, skip in zip(self.upconvs, self.decoders, reversed(skip_connections)):
            x = up(x)
            if x.shape != skip.shape:
                x = TF.resize(x, skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = dec(x)

        return self.final(x)


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(in_channels=3, num_classes=NUM_CLASSES).to(DEVICE)
print(f"Device: {DEVICE}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

### 1.5 Training

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

NUM_EPOCHS = 20
train_losses, val_losses = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    # --- Train ---
    model.train()
    running_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    train_loss = running_loss / len(train_ds)

    # --- Validate ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            val_loss += criterion(preds, masks).item() * imgs.size(0)
    val_loss /= len(val_ds)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

# Plot learning curves
plt.figure(figsize=(8, 4))
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training Curve")
plt.tight_layout()
plt.show()

### 1.6 Filter Visualization

In [ ]:
first_conv = model.encoders[0].net[0]  # first Conv2d layer
filters = first_conv.weight.data.cpu()  # [out_ch, in_ch, kH, kW]

n_filters = min(32, filters.shape[0])
fig, axes = plt.subplots(4, 8, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < n_filters:
        f = filters[i]  # [3, 3, 3]
        # Normalize to [0, 1] for display
        f = (f - f.min()) / (f.max() - f.min() + 1e-8)
        ax.imshow(f.permute(1, 2, 0).numpy())
    ax.axis("off")
plt.suptitle("First Conv Layer Filters (normalized)")
plt.tight_layout()
plt.show()

**Do the filters seem to detect useful patterns or features?**

> *Your answer here*

### 1.7 Evaluation

In [ ]:
def compute_iou_per_class(model, loader, num_classes, device):
    model.eval()
    intersection = torch.zeros(num_classes)
    union = torch.zeros(num_classes)
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            preds = model(imgs).argmax(dim=1)  # [B, H, W]
            for c in range(num_classes):
                pred_c = preds == c
                true_c = masks == c
                intersection[c] += (pred_c & true_c).sum().item()
                union[c] += (pred_c | true_c).sum().item()
    iou = intersection / (union + 1e-8)
    return iou


iou_per_class = compute_iou_per_class(model, val_loader, NUM_CLASSES, DEVICE)
print("IoU per class:")
for i, cls in CLASSES.items():
    print(f"  {cls}: {iou_per_class[i]:.4f}")
print(f"\nMean IoU: {iou_per_class.mean():.4f}")

plt.figure(figsize=(8, 4))
plt.bar([CLASSES[i] for i in range(NUM_CLASSES)], iou_per_class.numpy())
plt.ylabel("IoU")
plt.title("IoU per class")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# Qualitative evaluation: show predictions vs ground truth
model.eval()
imgs_b, masks_b = next(iter(val_loader))
with torch.no_grad():
    preds_b = model(imgs_b.to(DEVICE)).argmax(dim=1).cpu()

n_show = 3
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
for i in range(n_show):
    axes[i, 0].imshow(imgs_b[i].permute(1, 2, 0).numpy())
    axes[i, 0].set_title("Image")
    axes[i, 0].axis("off")
    axes[i, 1].imshow(masks_b[i].numpy(), cmap="tab10", vmin=0, vmax=NUM_CLASSES - 1)
    axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis("off")
    axes[i, 2].imshow(preds_b[i].numpy(), cmap="tab10", vmin=0, vmax=NUM_CLASSES - 1)
    axes[i, 2].set_title("Prediction")
    axes[i, 2].axis("off")
plt.tight_layout()
plt.show()

**Does the model perform better on certain classes? Are there classes that are harder to predict?**

> *Your answer here*

---
## Task 2 [4pkt]

Implement the **SeqScan** clustering algorithm based on the pseudocode on page 16 of the following paper:  
https://arxiv.org/pdf/1805.02102

Test your algorithm using one of the two options:
- Create a few small sample datasets manually (worth 1 point), or
- Generate your own synthetic dataset, similar to the example shown in Figure 12 of the paper (worth 2 points).

### 2.1 SeqScan Implementation

In [ ]:
from dataclasses import dataclass, field
from typing import List, Optional


def euclidean_dist(p, q):
    return np.sqrt(sum((a - b) ** 2 for a, b in zip(p, q)))


def seqscan(sequence, eps: float, min_pts: int, dist_fn=euclidean_dist):
    """
    SeqScan clustering for ordered sequences.

    Parameters
    ----------
    sequence : list of points  (each point is a tuple/list of coordinates)
    eps      : neighbourhood radius
    min_pts  : minimum number of points to form a dense region
    dist_fn  : distance function

    Returns
    -------
    labels : list[int]  -1 == noise, 0..K cluster indices
    """
    n = len(sequence)
    labels = [-1] * n
    cluster_id = 0

    # TODO: implement SeqScan as per pseudocode (p.16 of arxiv:1805.02102)
    # Hint: SeqScan is a sequential variant of DBSCAN that exploits order
    # to efficiently find dense contiguous sub-sequences.

    return labels


# Quick sanity check
toy = [(0,), (0.1,), (0.2,), (5,), (5.1,), (5.2,), (10,)]
labels = seqscan(toy, eps=0.5, min_pts=2)
print("Toy labels:", labels)

### 2.2 Test on Synthetic Dataset

In [ ]:
# --- Option A: small manual dataset ---
manual_seq = [
    (1.0, 2.0), (1.1, 2.1), (1.2, 1.9),  # cluster 0
    (8.0, 8.0),                            # noise
    (5.0, 5.0), (5.1, 4.9), (5.0, 5.1),  # cluster 1
]

labels_manual = seqscan(manual_seq, eps=0.3, min_pts=2)
print("Manual labels:", labels_manual)

xs, ys = zip(*manual_seq)
plt.figure(figsize=(6, 5))
plt.scatter(xs, ys, c=labels_manual, cmap="tab10", s=80, edgecolors="k")
plt.title("SeqScan – manual dataset")
plt.colorbar(label="cluster id (-1=noise)")
plt.show()

In [ ]:
# --- Option B: synthetic dataset inspired by Figure 12 of the paper ---
rng = np.random.default_rng(42)

def make_synthetic_sequence(n_clusters=4, cluster_size=30, noise_frac=0.1, spread=0.3, gap=3.0):
    points = []
    t = 0.0
    for _ in range(n_clusters):
        center = rng.uniform(0, 10, size=2)
        cluster_pts = center + rng.normal(0, spread, size=(cluster_size, 2))
        # order within cluster by arc-length approximation
        order = np.argsort(cluster_pts[:, 0])
        points.extend(cluster_pts[order].tolist())
        # add gap (noise) between clusters
        n_noise = int(cluster_size * noise_frac)
        noise_pts = center + rng.uniform(gap, gap * 3, size=(n_noise, 2))
        points.extend(noise_pts.tolist())
    return [tuple(p) for p in points]


synth_seq = make_synthetic_sequence()
labels_synth = seqscan(synth_seq, eps=0.5, min_pts=5)

xs, ys = zip(*synth_seq)
plt.figure(figsize=(8, 6))
plt.scatter(xs, ys, c=labels_synth, cmap="tab10", s=20, edgecolors="none")
plt.title("SeqScan – synthetic dataset (Fig. 12 style)")
plt.colorbar(label="cluster id (-1=noise)")
plt.show()

**Discussion of SeqScan results:**

> *Your answer here*

---
## Task 3 [3pkt]

**a)** [0.5 points] What is the time complexity of the approximate algorithm used for trajectory partitioning in the TRACLUS algorithm? Provide a justification for your answer.

**b)** [0.5 points] Give an example where the approximate algorithm for trajectory partitioning fails to find the optimal partitioning.

**c)** [1 point] Let's consider hurricane trajectories. Intuitively, stronger hurricanes should have higher weights when calculating the loss function \(N_\varepsilon\). How could you modify a density-based clustering algorithm for line segmentation to take trajectory weights into account?

**d)** [1 point] Is the original distance function used in density-based clustering for line segmentation a proper metric? Use the triangle inequality to provide a counterexample if it is not.

### 3a) Time Complexity of Approximate Trajectory Partitioning

> *Your answer here*
>
> **Justification:**

### 3b) Failure Case of Approximate Partitioning

> *Your example here*

### 3c) Weighting Hurricanes in Density-Based Clustering

> *Your answer here*

### 3d) Triangle Inequality Counterexample

> *Your answer / counterexample here*